In [1]:
import numpy as np
from time import sleep

from orchid.instrument import InstrumentAdapter
from orchid.parameter import DataKind, Parameter, Readout
from orchid.context import ExperimentContext
from orchid.procedure import ErrorPolicy, MonitorProcedure, Procedure, Sweep, WriteMode
from orchid.runner import ExperimentRunner
from orchid.plotting import LivePlotter, PlotSpec

In [2]:
class DummyVoltageSource:
    def __init__(self):
        self._voltage = 0.0

    @property
    def voltage(self):
        return self._voltage

    @voltage.setter
    def voltage(self, v):
        self._voltage = float(v)


class DummyLockin:
    """Lorentzian peak at V=0.5, with noise."""
    def __init__(self, source: DummyVoltageSource):
        self._src = source

    def read_x(self) -> float:
        v = self._src.voltage
        return 1.0 / (1.0 + ((v - 0.5) / 0.05) ** 2) + np.random.normal(0, 0.01)

    def read_y(self) -> float:
        return np.random.normal(0, 0.01)


class DummyGateSource:
    def __init__(self):
        self._vg1 = 0.0
        self._vg2 = 0.0

    @property
    def vg1(self): return self._vg1
    @vg1.setter
    def vg1(self, v): self._vg1 = v

    @property
    def vg2(self): return self._vg2
    @vg2.setter
    def vg2(self, v): self._vg2 = v


class DummyConductance:
    def __init__(self, gates: DummyGateSource):
        self._g = gates

    def read(self) -> float:
        # Gaussian peak in 2D gate space
        return np.exp(-((self._g.vg1 - 0.3)**2 + (self._g.vg2 - 0.7)**2) / 0.01) \
               + np.random.normal(0, 0.005)


class DummyVNA:
    def __init__(self):
        self._freq_center = 5e9  # Hz
        self.freqs = np.linspace(4e9, 6e9, 201)

    @property
    def freq_center(self): return self._freq_center
    @freq_center.setter
    def freq_center(self, f): self._freq_center = f

    def get_s21(self) -> np.ndarray:
        """Returns a resonance dip trace."""
        return -10 * np.exp(-((self.freqs - self._freq_center) / 50e6)**2) \
               + np.random.normal(0, 0.1, len(self.freqs))


In [3]:
# Instruments
vs = DummyVoltageSource()
li = DummyLockin(vs)
gs = DummyGateSource()
vna = DummyVNA()
cond = DummyConductance(gs)

# Context
ctx = ExperimentContext(data_root="./data")
ctx.add_instrument("vsrc", vs)
ctx.add_parameter("V", instrument="vsrc", attr="voltage", unit="V")
ctx.add_readout("X", kind=DataKind.SCALAR, get_func=li.read_x, unit="V")
ctx.add_readout("S21", kind=DataKind.TRACE, get_func=vna.get_s21, shape=(201,), unit="dB")

def Vmag():
    return np.sqrt(li.read_x()**2 + li.read_y()**2)
ctx.add_readout("Vmag", kind="scalar", get_func=Vmag, unit="V")
ctx

ExperimentContext(1 instruments, 1 parameters, 3 readouts)

In [4]:
ctx.snapshot()

Name    Type    Value                 Unit
------  ------  --------------------  ------
V       param   0.0                   V
X       scalar  0.03036111300685828   V
S21     trace   [...]                 dB
Vmag    scalar  0.030750215539574265  V


### **1D scan**

In [22]:
proc1 = Procedure(
    name="sweep",
    context=ctx,
    sweeps=[
        Sweep("V", np.linspace(0.1, 1, 101)),
    ],
    readouts=["X",'Vmag'],
    settle_time=0.1,          # 10ms settle after each set
    tags=["transport", "1d"],
    metadata={"field": "0T"},
    write_mode=WriteMode.SWEEPWISE
)

In [23]:
plotter = LivePlotter([
    PlotSpec(x="V", y="X", update_every="point"),
    PlotSpec(x="V", y="Vmag", update_every="point")
    ]
)

In [24]:
runner = ExperimentRunner()

In [25]:
ctx.snapshot()

Name    Type    Value                 Unit
------  ------  --------------------  ------
V       param   0.0                   V
X       scalar  0.006720388375775351  V
S21     trace   [...]                 dB
Vmag    scalar  0.002326747531742358  V


In [26]:
runner.run(proc1, plotter=plotter)

Live plot server started at http://localhost:8050


sweep: 100%|██████████| 101/101 [00:10<00:00,  9.67pt/s]

Experiment 'sweep' completed. Data saved to: data/0001


PosixPath('data/0001')

In [27]:
plotter.stop()

Live plot server stopped.


### **Time Series**

In [5]:
monitor = MonitorProcedure(
    name="stability_check",
    context=ctx,
    readouts=["X","Vmag"],
    interval=1.0,       # 1 second between reads
    duration=None,     # 1 hour
    tags=["stability"],
)
runner = ExperimentRunner()
plotter = LivePlotter([PlotSpec(x="_time", y="Vmag")], height=700, width=1400)

In [27]:
runner.is_monitoring

False

In [28]:
plotter.is_running

False

In [ ]:
runner.run_monitor(monitor, plotter=plotter, background=True)

Monitor 'stability_check' running in background. Use runner.stop_monitor() to stop.


Live plot server started at http://localhost:8050
Monitor 'stability_check' completed. Data saved to: data\0001


In [23]:
ctx["V"] = 0.46

In [24]:
runner.stop_monitor()

Monitor stopped. Data saved to: data\0001


WindowsPath('data/0001')

In [15]:
plotter.stop()

Live plot server stopped.
